# 3. `src/make_submission.py`: train the final models and write the prediction file

This is the project's **entry point**. Run from the project root:

```bash
.venv/bin/python src/make_submission.py
```

For each task it:

1. takes the feature function from `cv.TASK_FEATURES` and the model from `models.TASK_MODELS` (notebooks 1 and 2);
2. trains on **train + validation** (all 330 labelled flows) with seed 0;
3. predicts the 81 **test** flows and checks the output (every test flow once, only the 6 allowed app names);
4. writes `submissions/submission.json` (official format) and `submissions/submission.zip` (one CSV per task).

Each task reads only its own folder (Task 1 also reads the Task 2 feature files, which is allowed and disclosed).

> Code cells marked **from `src/make_submission.py`** are copied from that file at commit `3b4c9c2`; the `.py` file is the one the project actually runs. Cells marked **demo** are extra examples for reading only.
In this notebook `cv` and `models` are imported from `src/` rather than copied again.

In [1]:
# Setup: find the project root (the folder that holds data/official) and make src/ importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "official").exists())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

project root: <project root>


In [2]:
# from src/make_submission.py (OUT changed: see comment)
import io
import json
import zipfile
from pathlib import Path

import pandas as pd

from cv import DATA, TASK_FEATURES, TASKS, load_task
from models import TASK_MODELS

# In src/make_submission.py: ROOT = Path(__file__).resolve().parent.parent; OUT = ROOT / "submissions".
# Here OUT is a temporary folder, so running this notebook never overwrites the real submission files.
import tempfile
OUT = Path(tempfile.mkdtemp(prefix="walkthrough-submission-"))
ALLOWED = {"facebook", "google", "instagram", "mozilla", "x", "youtube"}
SEED = 0

## Training and predicting

`fit_predict` does steps 1 to 3 for one task and returns `{flow_id: app}` plus the fitted model. `load_test` reads
the test features (the test labels are not public and are never used).

In [3]:
# from src/make_submission.py
def load_test(task):
    return pd.read_csv(DATA / task / "test_features.csv")


def fit_predict(task, train_splits=("train", "validation"), predict_split="test", seed=SEED):
    """Fit the task's model (models.TASK_MODELS) on `train_splits` of one task and predict `predict_split` of it."""
    X_raw, y = load_task(task, train_splits)
    features = TASK_FEATURES[task]
    model = TASK_MODELS[task](seed).fit(features(X_raw), y)
    X_pred = load_test(task) if predict_split == "test" else load_task(task, (predict_split,))[0]
    pred = model.predict(features(X_pred))
    return dict(zip(X_pred.flow_id, pred)), model

## Checking and writing

- `check`: the predictions must cover exactly the test flows and use only the six allowed app names.
- `write_submission`: the JSON has one object per task (`flow_id → app`, sorted); the ZIP has `task1_static.csv` and
  `task2_realtime_prefix5.csv` with columns `flow_id, app`.

In [4]:
# from src/make_submission.py
def check(predictions, task, split="test"):
    ids = set((load_test(task) if split == "test" else load_task(task, (split,))[0]).flow_id)
    assert set(predictions) == ids, f"{task}: missing {len(ids - set(predictions))}, extra {len(set(predictions) - ids)}"
    assert set(predictions.values()) <= ALLOWED, f"{task}: invalid labels {set(predictions.values()) - ALLOWED}"


def write_submission(preds, json_path, zip_path):
    json_path.write_text(json.dumps({t: dict(sorted(preds[t].items())) for t in TASKS}, indent=2) + "\n")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for t in TASKS:
            buf = io.StringIO()
            pd.DataFrame(sorted(preds[t].items()), columns=["flow_id", "app"]).to_csv(buf, index=False)
            zf.writestr(f"{t}.csv", buf.getvalue())

## Running it

This is the script's `__main__` block. It writes to the temporary `OUT` folder.

In [5]:
# from src/make_submission.py (the __main__ block)
OUT.mkdir(exist_ok=True)
preds = {}
for task in TASKS:
    preds[task], _ = fit_predict(task)
    check(preds[task], task)
    print(task, len(preds[task]), "predictions:", pd.Series(preds[task]).value_counts().to_dict())
write_submission(preds, OUT / "submission.json", OUT / "submission.zip")
print("wrote", OUT / "submission.json", "and", OUT / "submission.zip")

task1_static 81 predictions: {'google': 33, 'x': 20, 'youtube': 13, 'mozilla': 9, 'facebook': 4, 'instagram': 2}


task2_realtime_prefix5 81 predictions: {'google': 32, 'x': 24, 'youtube': 12, 'mozilla': 7, 'instagram': 4, 'facebook': 2}
wrote /var/folders/02/cf_g4yv50y71txmpyc6myf0m0000gn/T/walkthrough-submission-5_wwf5yu/submission.json and /var/folders/02/cf_g4yv50y71txmpyc6myf0m0000gn/T/walkthrough-submission-5_wwf5yu/submission.zip


In [6]:
# demo: the file just built matches the production submission in the repository
import json
built = json.loads((OUT / "submission.json").read_text())
production = json.loads((ROOT / "submissions" / "production-100-cross-task" / "submission.json").read_text())
for task in TASKS:
    diff = [f for f in built[task] if built[task][f] != production[task].get(f)]
    print(f"{task}: {len(built[task])} flows, {len(diff)} differ from submissions/production-100-cross-task/")

task1_static: 81 flows, 0 differ from submissions/production-100-cross-task/
task2_realtime_prefix5: 81 flows, 0 differ from submissions/production-100-cross-task/
